---
title: 'Quickstart: Inspecting Agent Tool-Selection Features'
label: quickstart
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/575-lab/scipy_proceedings/blob/2026/papers/575-lab/quickstart_colab.ipynb)

This is an open-source example of the pipeline applied to a small home-repair dataset. It uses a JumpReLU SAE trained specifically on this scenario — **not** the SAE reported in the paper — to illustrate the workflow end to end. The notebook captures a hidden-state activation from `google/gemma-4-E4B-it` at the decision token, loads that pretrained SAE with `SAE.from_pretrained`, and describes the top features driving the agent's tool selection. It requires only `transformers` and `kiji-inspector` and runs on CPU.


## Setup

Install the two dependencies. The quickstart runs end-to-end on **CPU** — no GPU required.

In [ ]:
!pip install -q kiji-inspector==0.5.0rc2 transformers==5.12.1

## 1. Capture the decision-token activation

We give `google/gemma-4-E4B-it` a home-repair request together with a system prompt that
offers four tools (`manual_check`, `parts_search`, `tutorial_search`, `pro_quote`). A forward
hook on layer 8 captures the residual-stream activation at the **decision token** — the
position where the model commits to a tool. This single hidden-state vector is what we
interpret in the rest of the notebook.

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

LAYER_INDEX = 8
MODEL_ID = "google/gemma-4-E4B-it"
PROMPT = "My dishwasher is smelly, what is the first element I should review?"

SYSTEM_PROMPT = (
    "You are a home repair advisor helping homeowners diagnose appliance and household "
    "problems. You analyze diagnostic data from various tools to determine whether a problem "
    "requires professional service or can be safely handled as a DIY repair. Always consider "
    "safety first, then cost, difficulty, and warranty implications. Ground your advice in "
    "specific details from the data provided.\n\n"
    "Available tools:\n"
    "- manual_check: Look up appliance troubleshooting guides, error codes, manufacturer-recommended diagnostic steps, and safety warnings\n"
    "- parts_search: Search for replacement parts with pricing, availability, compatibility information, and estimated shipping times\n"
    "- tutorial_search: Find video tutorials and step-by-step repair guides with difficulty ratings, required tools, and estimated completion times\n"
    "- pro_quote: Get professional repair service quotes including labor costs, typical turnaround times, service warranties, and urgency assessment\n\n"
    "Respond with the single most appropriate tool to call for the user's request."
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user", "content": [{"type": "text", "text": PROMPT}]},
]
prompt = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = processor(text=prompt, return_tensors="pt")
inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

captured = {}

def hook(_module, _inputs, output):
    hidden = output[0] if isinstance(output, tuple) else output
    captured["tensor"] = hidden.detach().cpu()

layer = model.model.language_model.layers[LAYER_INDEX]
handle = layer.register_forward_hook(hook)
try:
    with torch.inference_mode():
        model(**inputs)
finally:
    handle.remove()

# Last-token activation at LAYER_INDEX, no normalization.
hidden_state = captured["tensor"][0, -1]
print(hidden_state.shape)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


torch.Size([2560])


Sanity-check the shapes: the tokenized prompt, and the single captured
activation vector (`d_model = 2560`) at the decision token.

In [4]:
inputs['input_ids'].shape

torch.Size([1, 213])

In [5]:
hidden_state.shape

torch.Size([2560])

## 2. Decompose the activation into features

A pretrained **JumpReLU Sparse Autoencoder (SAE)** decomposes the dense activation into a
sparse set of **monosemantic features** drawn from the model's own learned vocabulary.
`SAE.from_pretrained` loads both the SAE weights and the human-readable feature labels, and
`describe` returns the features firing most strongly on the decision token.

In [7]:
from kiji_inspector import SAE

sae, feature_descriptions = SAE.from_pretrained(
    repo_id="575-lab/kiji-inspector-google-gemma-4-E4B-it-home-repair",
    layer=8,
)

# Extract the activation for the first sequence, last token
last_token_act = hidden_state

# Describe the top features activating on this token using the descriptions dictionary
sae.describe(last_token_act, feature_descriptions)


[(331,
  {'label': 'Home appliance malfunction and repair cost inquiry',
   'description': 'Detects user queries regarding malfunctioning household appliances (dishwashers, refrigerators, washing machines) specifically involving error codes, leaks, or assessing repair costs versus replacement due to expired warranties.',
   'confidence': 'high',
   'mean_activation': 5.917806,
   'max_activation': 6.46875,
   'frac_nonzero': 1.0,
   'top_examples': ['My dishwasher is actively leaking water onto the kitchen floor right now, I need to know the immediate safety steps and error codes to stop the damage.',
    'My dishwasher is leaking water onto the floor right now, I need to know the immediate safety steps and error codes to stop the damage.',
    'My 10-month-old refrigerator is displaying an E4 error code; since the warranty was voided by a power surge, what are the costs for parts and labor to fix it?',
    'My dishwasher has a small drip under the door that I noticed this morning, I n

## 3. Read the decision report

Mapping the top active features to their labels yields a human-readable account of the
factors driving the agent's tool selection:

In [8]:
results = sae.describe(last_token_act, feature_descriptions)

for feature_id, desc, activation in results:
    print(f"Feature ID: {feature_id} | Activation: {activation:.2f}")
    if isinstance(desc, dict):
        print(f"  Label: {desc.get('label', 'N/A')}")
        print(f"  Description: {desc.get('description', 'N/A')}")
        print(f"  Confidence: {desc.get('confidence', 'N/A')}")
    else:
        print(f"  Label: {desc}")
    print("-" * 50)

Feature ID: 331 | Activation: 6.30
  Label: Home appliance malfunction and repair cost inquiry
  Description: Detects user queries regarding malfunctioning household appliances (dishwashers, refrigerators, washing machines) specifically involving error codes, leaks, or assessing repair costs versus replacement due to expired warranties.
  Confidence: high
--------------------------------------------------
Feature ID: 181 | Activation: 6.24
  Label: DIY appliance component replacement and repair
  Description: This feature detects user intent to perform self-service repairs involving the replacement of specific mechanical parts like valves, gaskets, or seals in household appliances.
  Confidence: high
--------------------------------------------------
Feature ID: 579 | Activation: 6.07
  Label: unknown
--------------------------------------------------
Feature ID: 648 | Activation: 6.02
  Label: DIY appliance part replacement and repair
  Description: The feature detects requests for id

## Next steps

This quickstart shows the inference path on a single prompt. The full pipeline — contrastive
feature analysis (which features distinguish one tool choice from another), token-level
fuzzing evaluation, and causal feature ablation — is described in the paper and available in
the [project repository](https://github.com/dataiku/kiji-inspector).